In [1]:
import pandas as pd

X_train = pd.read_csv("../data/processed/X_train.csv", index_col=0)
X_val   = pd.read_csv("../data/processed/X_val.csv", index_col=0)
X_test  = pd.read_csv("../data/processed/X_test.csv", index_col=0)

In [3]:
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_val = pd.read_csv("../data/processed/y_val.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [4]:
X_train.shape, X_val.shape, X_test.shape, y_train.shape, y_val.shape, y_test.shape

((97472, 13), (20887, 13), (20887, 13), (97472,), (20887,), (20887,))

In [5]:
from sklearn.preprocessing import StandardScaler

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [8]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_scaled, y_train)
y_val_pred = model.predict(X_val_scaled)

In [10]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_pred))
print("F1 Score:", f1_score(y_val, y_val_pred, average="macro"))

Accuracy: 0.6834873366208647
Balanced Accuracy: 0.6756339912441411
F1 Score: 0.37356918025579


## Baseline results (Logistic Regression)

On the validation set:

- Accuracy: 0.683
- Balanced accuracy: 0.676
- Macro F1: 0.374

Compared to a dummy classifier that always predicts `optic` (the majority class):

- Accuracy: 0.56
- Balanced accuracy: 0.10

The logistic regression clearly beats the dummy baseline on balanced accuracy, showing it is picking up real signal across the rare classes, not just exploiting the class imbalance.

Macro F1 is much lower than balanced accuracy because `class_weight="balanced"` only helps with **recall**, not precision. Reweighting the loss pushes the model to predict rare classes more often, so it catches more of their true instances (higher recall). But it does not stop the model from over-predicting those same rare classes on examples that actually belong elsewhere, so precision on the rare classes drops. Since F1 is the harmonic mean of precision and recall, a low precision drags the macro F1 down even when recall (and therefore balanced accuracy, which only looks at recall per class) is high.